# Trace widths: comparing estimators

Which estimator should `imageQualityQa` use for the cross-dispersion trace width on a calexp
(PIPE2D-1918)? This notebook measures the same quartz detectors several ways and compares them:

| Label | Estimator | Source |
|---|---|---|
| `jointFit` | each fiber fitted with its neighbours, per sampled row | `pfs.drp.qa.crossDispersion.measureImageWidths` |
| `expMoment`, `expFit` | profiles built from **this calexp** with drp_stella's `FiberProfileSet.fromImages` (`fitSwathProfiles`), read as a second moment and as a Gaussian fit | drp_stella, as `buildFiberProfiles` does for the calibration |
| `calibMoment`, `calibFit` | the `fiberProfiles` calibration, read the same two ways | today's fallback |
| `kernelFit` | optional: the calibration profiles broadened by a `FiberKernel` measured on this calexp | drp_stella `MeasureFiberKernelTask` |

Every FWHM here **includes the pixel response** (the profile as the pixels record it), as
`SdssShape` moments and `fiberProfiles` do. "Moment" is drp_stella's
`FiberProfile.calculateStatistics().width`; "Fit" is `crossDispersion.profileWidth`, a Gaussian plus
a constant.

It also answers two side questions: why 133040 r2 has read wider than its calibration while b2 has
not (does the *exposure's* own profile agree with the joint fit or with the calibration?), and what
reading `lines` costs on a trace visit.

The notebook only reads the Butler. **Start Jupyter from a shell with the stack set up**:

```bash
source /path/to/stack/loadLSST.bash
setup -r /path/to/drp_stella
setup -r /path/to/drp_qa
cd /path/to/drp_qa/docs
jupyter lab trace-widths.ipynb
```

Set the inputs in section 1, then run all cells. Commit it with its outputs: they are the record
behind the choice.

In [ ]:
!eups list --setup drp_stella

In [ ]:
!eups list --setup drp_qa

## 1. Inputs

The collections must hold each visit's `calexp` and `detectorMap`, as a `docs/qa-thresholds.ipynb`
run does, and the calibrations. Results are cached per detector in `FILES`; delete a file to
measure it again.

In [ ]:
# Likely to change
COLLECTIONS = ["u/wtg/qa-thresholds/004", "PFS/defaults"]
DETECTORS = {
    # Run25 known-good quartz, every arm of every spectrograph.
    "good": [(133040, arm, sm) for arm in "brn" for sm in (1, 2, 3, 4)],
    # Run27 known-bad: SM1 defocused, expect medFwhm FAIL.
    "defocused": [(visit, arm, 1) for visit in (140032, 140139) for arm in "brn"],
    # Run25 known-bad: obstructed (M1 cover closed); for the lines cost and nLines.
    "obstructed": [(150116, arm, 1) for arm in "brn"],
}
DO_KERNEL = (
    False  # True: also measure a FiberKernel per detector (slow; drp_stella code still in development)
)

# Rarely change
BUTLER = "/work/datastore"
FILES = "~/.cache/drp_qa/trace-widths"
ROW_STRIDE = 50  # imageQualityQa's profileYStride
BAD_PLANES = ["BAD", "SAT", "CR", "NO_DATA"]
PROFILE = {"radius": 5, "oversample": 10, "swathSize": 300}  # buildFiberProfiles defaults

## Set up

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lsst.daf.butler import Butler

from pfs.drp.qa.crossDispersion import FWHM_FACTOR, measureImageWidths, profileWidth
from pfs.drp.stella import FiberProfile, FiberProfileSet

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

butler = Butler(BUTLER, collections=COLLECTIONS, writeable=False)
FILES = Path(FILES).expanduser()
FILES.mkdir(parents=True, exist_ok=True)
print(f"files: {FILES}")

## 2. Measurement

`profileTable` reads one `FiberProfileSet` both ways, one row per fiber and swath. `exposureProfiles`
builds a set from the calexp the way `BuildFiberProfilesTask.runMultiple` does for a calibration,
without its iterations: centers from the visit's detector map, fluxes from an extraction with
Gaussian profiles at the calibration's median sigma, then `fromImages` (which fits all fibers of a
swath together, so neighbours are separated).

In [ ]:
def profileTable(profiles, label):
    """One row per fiber and swath: second-moment and Gaussian-fit FWHM."""
    frames = []
    for fiberId in profiles:
        profile = profiles[fiberId]
        moment = np.ma.filled(
            np.ma.masked_invalid(np.ma.asarray(profile.calculateStatistics().width)), np.nan
        )
        fit = profileWidth(profile.index, profile.profiles)["sigma"]
        frames.append(
            pd.DataFrame(
                {
                    "fiberId": fiberId,
                    "y": profile.rows,
                    f"{label}Moment": FWHM_FACTOR * moment,
                    f"{label}Fit": FWHM_FACTOR * fit,
                }
            )
        )
    return pd.concat(frames, ignore_index=True)


def exposureProfiles(calexp, detectorMap, calib):
    """Profiles of this calexp, built with drp_stella's fromImages."""
    height = calexp.getHeight()
    fiberId = np.array([ff for ff in detectorMap.fiberId if ff in calib], dtype=int)
    widths = np.concatenate(
        [np.ma.filled(np.ma.asarray(calib[ff].calculateStatistics().width), np.nan) for ff in fiberId]
    )
    sigma = float(np.nanmedian(widths))
    gaussian = FiberProfileSet.makeEmpty(
        calib.identity, calexp.getInfo().getVisitInfo(), calexp.getMetadata()
    )
    for ff in fiberId:
        gaussian[ff] = FiberProfile.makeGaussian(sigma, height, PROFILE["radius"], PROFILE["oversample"])
    badBitMask = calexp.mask.getPlaneBitMask(BAD_PLANES)
    rows = np.arange(height, dtype=float)
    norm = np.zeros((len(fiberId), height), dtype=np.float32)
    index = {ff: ii for ii, ff in enumerate(fiberId)}
    for spectrum in gaussian.extractSpectra(calexp.maskedImage, detectorMap, badBitMask):
        flux = np.array(spectrum.flux, dtype=float)
        bad = (spectrum.mask.array[0] & badBitMask) != 0
        if bad.any() and (~bad).any():
            flux[bad] = np.interp(rows[bad], rows[~bad], flux[~bad], 0.0, 0.0)
        norm[index[spectrum.fiberId]] = flux
    centers = np.array([detectorMap.getXCenter(int(ff)) for ff in fiberId])
    return FiberProfileSet.fromImages(
        calib.identity,
        [calexp.maskedImage],
        fiberId,
        [centers],
        [norm],
        PROFILE["radius"],
        PROFILE["oversample"],
        PROFILE["swathSize"],
        maskPlanes=BAD_PLANES,
    )


def kernelProfiles(calexp, detectorMap, calib):
    """Return the calibration profiles broadened by a FiberKernel measured on this calexp."""
    from pfs.drp.stella.measureFiberKernel import MeasureFiberKernelTask

    task = MeasureFiberKernelTask()
    kernel = task.run(calexp.maskedImage, detectorMap, calib)
    return task.convolveProfiles(
        kernel, calib, detectorMap, calib.identity, calexp.getInfo().getVisitInfo(), calexp.getMetadata()
    )


def measure(visit, arm, spectrograph):
    """Measure one detector every way; cached as two parquet files."""
    stem = f"{visit}-{arm}{spectrograph}"
    rowFile, profFile = FILES / f"jointFit-{stem}.parquet", FILES / f"profiles-{stem}.parquet"
    if rowFile.exists() and profFile.exists():
        return pd.read_parquet(rowFile), pd.read_parquet(profFile)
    dataId = {"instrument": "PFS", "visit": visit, "arm": arm, "spectrograph": spectrograph}
    timing = {}
    calexp = butler.get("calexp", dataId)
    detectorMap = butler.get("detectorMap", dataId)
    calib = butler.get("fiberProfiles", dataId)

    start = time.perf_counter()
    rows = np.arange(ROW_STRIDE // 2, calexp.getHeight(), ROW_STRIDE)
    fiberIds = np.asarray(detectorMap.fiberId, dtype=np.int32)
    yy = np.repeat(rows[:, None].astype(float), len(fiberIds), axis=1)
    ff = np.repeat(fiberIds[None, :], len(rows), axis=0)
    xCenters = detectorMap.getXCenter(ff.ravel(), yy.ravel()).reshape(yy.shape)
    wavelengths = detectorMap.findWavelength(ff.ravel(), yy.ravel()).reshape(yy.shape)
    badBits = calexp.mask.getPlaneBitMask(BAD_PLANES)
    joint = measureImageWidths(
        calexp.image.array[rows],
        calexp.variance.array[rows],
        (calexp.mask.array[rows] & badBits) != 0,
        rows,
        fiberIds,
        xCenters,
        wavelengths,
    )
    timing["jointFit"] = time.perf_counter() - start

    start = time.perf_counter()
    exposure = profileTable(exposureProfiles(calexp, detectorMap, calib), "exp")
    timing["expProfiles"] = time.perf_counter() - start
    table = exposure.merge(profileTable(calib, "calib"), on=["fiberId", "y"], how="outer")
    if DO_KERNEL:
        start = time.perf_counter()
        kernel = profileTable(kernelProfiles(calexp, detectorMap, calib), "kernel")
        timing["kernel"] = time.perf_counter() - start
        table = table.merge(kernel[["fiberId", "y", "kernelFit"]], on=["fiberId", "y"], how="outer")

    for frame in (joint, table):
        frame["visit"], frame["arm"], frame["spectrograph"] = visit, arm, spectrograph
    calibRef = butler.find_dataset("fiberProfiles", dataId)
    table.attrs = joint.attrs = {"calibRun": str(calibRef.run), **{k: f"{v:.1f}" for k, v in timing.items()}}
    joint.to_parquet(rowFile)
    table.to_parquet(profFile)
    print(f"{stem}: " + ", ".join(f"{k} {v:.0f} s" for k, v in timing.items()))
    return joint, table

In [ ]:
results = {}
for group, detectors in DETECTORS.items():
    for detector in detectors:
        try:
            results[detector] = (group, *measure(*detector))
        except Exception as error:  # a missing dataset should not stop the others
            print(f"{detector}: {type(error).__name__}: {error}")

## 3. Per-detector summary

Median FWHM per estimator (px), the joint fit's usable fraction, the calibration's run and the time
each measurement took.

In [ ]:
ESTIMATORS = ["jointFit", "expMoment", "expFit", "calibMoment", "calibFit"] + (
    ["kernelFit"] if DO_KERNEL else []
)


def summary(detector, group, joint, table):
    good = ~joint["flag"]
    row = dict(
        zip(["visit", "arm", "spectrograph"], detector, strict=True),
        group=group,
        usable=good.mean(),
        jointFit=joint.loc[good, "fwhm"].median(),
    )
    row.update({name: table[name].median() for name in ESTIMATORS[1:]})
    row.update(table.attrs)
    return row


summaryTable = pd.DataFrame([summary(det, *res) for det, res in results.items()])
summaryTable.round(3)

Each estimator against the calibration fit, per detector: ratios near 1 mean "reads like the calibration".

In [ ]:
ratios = summaryTable.set_index(["group", "visit", "arm", "spectrograph"])[ESTIMATORS].div(
    summaryTable.set_index(["group", "visit", "arm", "spectrograph"])["calibFit"], axis=0
)
ratios.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, arm in zip(axes, "brn", strict=True):
    data = summaryTable[summaryTable.arm == arm]
    labels = [f"{v}\n{a}{s}" for v, a, s in zip(data.visit, data.arm, data.spectrograph, strict=True)]
    for name in ESTIMATORS:
        ax.plot(labels, data[name], "o", label=name, alpha=0.8)
    ax.set_title(f"{arm} arm")
    ax.tick_params(axis="x", labelsize=7)
axes[0].set_ylabel("median FWHM, pixel response included (px)")
axes[-1].legend(fontsize=8)
fig.tight_layout()

## 4. Per-fiber agreement

The joint fit's per-fiber median against the exposure's own profile (`expFit`) and the calibration
(`calibFit`), for the known-good detectors. Agreement with `expFit` tests the method; a difference
that `expFit` shares with the joint fit is the exposure, not the estimator.

In [ ]:
def perFiber(joint, table):
    jf = joint[~joint["flag"]].groupby("fiberId")["fwhm"].median().rename("jointFit")
    return pd.concat([jf, table.groupby("fiberId")[["expFit", "calibFit"]].median()], axis=1).dropna()


goodDetectors = [det for det, res in results.items() if res[0] == "good"]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for det in goodDetectors:
    fibers = perFiber(*results[det][1:])
    for ax, name in zip(axes, ["expFit", "calibFit"], strict=True):
        ax.plot(fibers[name], fibers["jointFit"], ".", ms=2, alpha=0.4, label=f"{det[1]}{det[2]}")
for ax, name in zip(axes, ["expFit", "calibFit"], strict=True):
    lims = ax.get_xlim()
    ax.plot(lims, lims, "k:", lw=1)
    ax.set_xlabel(f"{name} FWHM (px)")
axes[0].set_ylabel("jointFit FWHM (px)")
axes[1].legend(markerscale=5, fontsize=7, ncol=2)
fig.tight_layout()

## 5. 133040 r2 against b2

The open question: the joint fit read r2 about 10 % wider than its calibration and b2 the same.

In [ ]:
summaryTable[(summaryTable.visit == 133040) & (summaryTable.spectrograph == 2)][
    ["arm", *ESTIMATORS, "calibRun"]
].round(3)

## 6. Known-good against defocused

Distributions of the joint fit's samples per arm: the defocused spectrograph must read wider.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, arm in zip(axes, "brn", strict=True):
    for group in ("good", "defocused"):
        samples = [
            res[1].loc[~res[1]["flag"], "fwhm"]
            for det, res in results.items()
            if res[0] == group and det[1] == arm
        ]
        if samples:
            ax.hist(
                pd.concat(samples), bins=np.linspace(2, 6, 81), histtype="step", density=True, label=group
            )
    ax.set_title(f"{arm} arm")
    ax.set_xlabel("jointFit FWHM (px)")
axes[0].legend()
fig.tight_layout()

## 7. The cost of reading `lines` on a trace visit

`nLines` (rows of `lines`) is the only metric that catches obstructed quartz (PIPE2D-1917), so the
task keeps reading it unless this is material.

In [ ]:
costs = []
for det, (group, _, _) in results.items():
    if det[1] != "r":
        continue
    dataId = {"instrument": "PFS", "visit": det[0], "arm": det[1], "spectrograph": det[2]}
    start = time.perf_counter()
    lines = butler.get("lines", dataId)
    elapsed = time.perf_counter() - start
    frame = lines.data if hasattr(lines, "data") else lines
    costs.append(
        {
            "visit": det[0],
            "arm": det[1],
            "spectrograph": det[2],
            "group": group,
            "rows": len(frame),
            "seconds": round(elapsed, 1),
            "memoryMB": round(frame.memory_usage(deep=True).sum() / 1e6),
        }
    )
pd.DataFrame(costs)

## 8. Conclusion

To be written from the outputs above: the chosen estimator and why, the r2 explanation, and the
`lines` decision. The outcome goes in the README and `docs/qa-domain-notes.md`.